# リークの 4 パターン（演習の答え合わせ）

**出典**　機械学習 4 分野の教科書 — 第 4 章 1-C アンサンブル・評価・前処理 ／ 08 演習と解答

**このノートのパス**　`ml-code/04_ensemble_eval/04_leakage_patterns.ipynb`

**このノートで確かめること**
教科書の演習に出てくる **4 つの断片**を、実際に動く形に補って並べる。
①誤り ②手で直した形 ③Pipeline で直した形 ④時系列での正しい分割。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, numpy` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

共通のデータとモデルを用意する（教科書の断片は「前後の定義は省略」となっている部分）。

In [3]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge

X, y = make_classification(n_samples=800, n_features=20, n_informative=8,
                           random_state=0)
model = LogisticRegression(max_iter=1000)

## ① 誤りを含むコード — 全データで fit している

`StandardScaler` を**分割前の全データ**で fit しているため、テストデータの平均・分散が訓練側に漏れている。

In [4]:
scaler = StandardScaler().fit(X)           # 全データで fit ← ここが誤り
X_scaled = scaler.transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=0)
model.fit(X_tr, y_tr)
print("リークあり:", model.score(X_te, y_te))

リークあり: 0.89375


## ② 手で直した形 — 訓練データだけで fit する

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_tr)        # 訓練データだけで fit
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)            # テストには transform だけ

model.fit(X_tr_s, y_tr)
print("リークなし:", model.score(X_te_s, y_te))

リークなし: 0.89375


## ③ Pipeline で直した形 — 各 fold の内側で fit される

**交差検証では②の手作業では不十分**である。`cross_val_score` の外で fit すると、結局その fold の検証データを含めて標準化してしまう。

In [6]:
pipe = make_pipeline(StandardScaler(), model)
print("CV スコア:", cross_val_score(pipe, X, y, cv=5).round(4))
# 各 fold の内側で StandardScaler が fit される

CV スコア: 

[0.9125 0.9312 0.8938 0.9188 0.9188]


## ④ 時系列データでは、ランダムに分割してはいけない

`KFold` は未来のデータで学習して過去を予測してしまう。`TimeSeriesSplit` は常に過去→未来の順を守る。

In [7]:
# 時系列らしい回帰データを作る（トレンド + ノイズ）
rng = np.random.default_rng(0)
n = 400
t = np.arange(n)
Xt = np.c_[t, np.sin(t / 20), rng.normal(0, 1, n)]
yt = 0.02 * t + 2 * np.sin(t / 20) + rng.normal(0, 0.3, n)

pipe_r = make_pipeline(StandardScaler(), Ridge())
tscv = TimeSeriesSplit(n_splits=5, test_size=30)   # 直近 30 日ずつ検証
s = cross_val_score(pipe_r, Xt, yt, cv=tscv, scoring="neg_root_mean_squared_error")
print("TimeSeriesSplit RMSE:", (-s).round(4))

for i, (tr, va) in enumerate(tscv.split(Xt)):
    print(f"  fold {i}: train [{tr.min()}..{tr.max()}]  valid [{va.min()}..{va.max()}]")

TimeSeriesSplit RMSE: [0.275  0.2965 0.2697 0.3451 0.3114]
  fold 0: train [0..249]  valid [250..279]
  fold 1: train [0..279]  valid [280..309]
  fold 2: train [0..309]  valid [310..339]
  fold 3: train [0..339]  valid [340..369]
  fold 4: train [0..369]  valid [370..399]


## 読み取りどころ

①と②のスコア差は、このデータでは小さいかほぼ無いはずである。**リークの怖さは「必ず大きく効く」ことではなく、「効いたかどうかが分からない」こと**にある。
標準化のような弱いリークは差が出ないこともあるが、ターゲットエンコーディングや時系列の取り違えでは、AUC が 0.05〜0.2 平気で動く。

④の fold の範囲表示に注目。**train の終端が valid の始端より必ず前にある。**これが `KFold` との決定的な違いである。